In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from lightgbm import LGBMRegressor

In [3]:
## featured data from parquet file 

featured_PATH = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "featured_data.parquet"
)
data = pd.read_parquet(featured_PATH)

data

,date,consumption,year,month,day,weekday,national_temp,hour,consumption_lag_24,consumption_lag_48,...,CDD,temp_lag_24,temp_lag_48,temp_rolling_mean_24,yearly_baseline,target_ratio,day_length,is_dark,hour_to_sunrise,is_covid
0,2017-01-01 00:00:00+03:00,27223.06,2017,1,1,Sunday,1.314326,0,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,8.166667,0
1,2017-01-01 01:00:00+03:00,25825.90,2017,1,1,Sunday,0.719622,1,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,7.166667,0
2,2017-01-01 02:00:00+03:00,24252.68,2017,1,1,Sunday,0.304513,2,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,6.166667,0
3,2017-01-01 03:00:00+03:00,22915.47,2017,1,1,Sunday,-0.786601,3,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,5.166667,0
4,2017-01-01 04:00:00+03:00,22356.99,2017,1,1,Sunday,-0.671492,4,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,9.383333,1,4.166667,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84691,2026-08-30 19:00:00+03:00,43487.20,2026,8,30,Sunday,25.720893,19,46146.69,50946.09,...,9.720893,23.592781,23.277993,21.967974,40589.056963,1.071402,13.166667,0,NaN,0
84692,2026-08-30 20:00:00+03:00,44306.44,2026,8,30,Sunday,24.377825,20,46276.80,50700.57,...,8.377825,22.749788,22.444577,21.980691,40588.471465,1.091602,13.166667,1,10.233333,0
84693,2026-08-30 21:00:00+03:00,43046.39,2026,8,30,Sunday,24.069101,21,44800.31,49012.85,...,8.069101,22.351889,21.814356,22.003089,40587.751648,1.060576,13.166667,1,9.233333,0
84694,2026-08-30 22:00:00+03:00,41560.60,2026,8,30,Sunday,23.356092,22,42736.17,47010.37,...,7.356092,21.627122,20.642315,22.044122,40586.758430,1.023994,13.166667,1,8.233333,0


In [4]:
# Check NaN 

data.isna().sum()

# Traget variable is target_ratio thus it can not contain NaN
# Thus the first 8789 lines shoul be dropped. The remaining NaN are in hours_to_sunrise which are daylight data.

data = data.dropna(subset=["target_ratio"])
data.isna().sum()

date                                0
consumption                         0
year                                0
month                               0
day                                 0
weekday                             0
national_temp                       0
hour                                0
consumption_lag_24                  0
consumption_lag_48                  0
consumption_lag_168                 0
consumption_rolling_mean_24         0
consumption_rolling_mean_168        0
is_weekend                          0
DoW                                 0
DoY                                 0
is_religious_holiday                0
is_arife                            0
is_national_holiday                 0
is_kopru                            0
bayram_day_index                    0
HDD                                 0
CDD                                 0
temp_lag_24                         0
temp_lag_48                         0
temp_rolling_mean_24                0
yearly_basel

In [5]:
# ---------Train-Val-Test Split-------------

#---Split the data train val test---
train = data[data["date"] < "2024-09-01"].copy()
val   = data[(data["date"] >= "2024-09-01") & (data["date"] < "2025-09-01")].copy()
test  = data[data["date"] >= "2025-09-01"].copy()
print(len(train), len(val), len(test))


exclude_cols = ["date", "weekday", "consumption", "target_ratio", "day"]
feature_cols = [col for col in data.columns if col not in exclude_cols]

categorical_cols = ["hour", "DoW", "bayram_day_index", "DoY", "month"]


X_train, y_train = train[feature_cols], train["target_ratio"]
X_val, y_val = val[feature_cols], val["target_ratio"]

for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype("category")

58411 8760 8736


C:\Users\Sultan Selin\AppData\Local\Temp\ipykernel_17856\3773287743.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[col] = X_train[col].astype("category")
C:\Users\Sultan Selin\AppData\Local\Temp\ipykernel_17856\3773287743.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val[col] = X_val[col].astype("category")


In [10]:
# -------- LightGBM Model -------------#
model = LGBMRegressor(
    objective = "regression",
    random_state=42,
    importance_type="gain"
)

model.fit(X_train, y_train, categorical_feature=categorical_cols)
importance_gain = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)###
importance_df = importance_gain.reset_index()
importance_df.columns = ["feature", "gain"]
importance_df

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003219 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4002
[LightGBM] [Info] Number of data points in the train set: 58411, number of used features: 27
[LightGBM] [Info] Start training from score 1.013554


,feature,gain
0,consumption_lag_168,4083.796738
1,consumption_lag_24,1651.747503
2,yearly_baseline,609.242490
3,DoY,405.157422
4,is_religious_holiday,302.855297
5,DoW,192.449505
6,is_weekend,158.391841
7,hour,93.888804
8,CDD,74.346468
9,is_arife,34.682452


In [8]:
# ------Consumption calc over target ratio ----------#

predicted_ratio = model.predict(X_val)
predicted_consumption = predicted_ratio * val["yearly_baseline"]

MAPE = ((val["consumption"] - predicted_consumption).abs() / val["consumption"]).mean() * 100
print("LightGBM Val MAPE (%):", MAPE)


val_copy = val.copy()
val_copy["predicted_consumption"] = predicted_consumption
val_copy["abs_pct_error"] = ((val_copy["consumption"] - val_copy["predicted_consumption"]).abs() / val_copy["consumption"]) * 100

print("Hourly MAPE:", val_copy.groupby("hour")["abs_pct_error"].mean())


LightGBM Val MAPE (%): 4.15127260726803
Hourly MAPE: hour
0     4.292855
1     4.305960
2     4.093051
3     3.847875
4     3.891381
5     3.913342
6     3.898581
7     4.291807
8     4.602832
9     4.492633
10    4.420253
11    4.432824
12    4.491895
13    4.579152
14    4.654678
15    4.576697
16    4.452996
17    4.078023
18    3.633929
19    3.495569
20    3.501147
21    3.741314
22    3.903718
23    4.038030
Name: abs_pct_error, dtype: float64


In [9]:
print("Weekday MAPE:", val_copy.groupby("weekday")["abs_pct_error"].mean())


Weekday MAPE: weekday
Friday       4.085675
Monday       4.594677
Saturday     4.219955
Sunday       5.080675
Thursday     3.786508
Tuesday      3.587868
Wednesday    3.685677
Name: abs_pct_error, dtype: float64


In [ ]:
# Drop some features based on importance gain, redundancy and relevancy

drop_features = ["month", "year", ]